<a href="https://colab.research.google.com/github/Molten-rock/AI-powered-password-breach/blob/main/Data_Breach_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

First, upload your CSV file. You can use the file upload functionality in Colab. Once uploaded, the file will be available in the Colab environment.

In [2]:
# This cell is no longer needed as the file already exists.
# from google.colab import files

# uploaded = files.upload("/content/Breached_data.csv")

# for fn in uploaded.keys():
#   print('User uploaded file "{name}" with length {length} bytes'.format(
#       name=fn, length=len(uploaded[fn])))

In [3]:
import pandas as pd
from pathlib import Path

DATA_PATH = "/content/Breached_data.csv"  # updated path to the uploaded file

p = Path(DATA_PATH)
print("File exists:", p.exists(), "size MB:", p.stat().st_size / (1024*1024))

# Many rockyou files are one password per line (no header). Read safely:
# Use error_bad_lines=False and warn_bad_lines=True to skip bad lines if any
df = pd.read_csv(DATA_PATH, header=None, names=["password"], encoding='utf-8', engine='python', on_bad_lines='warn')
print("Rows:", len(df))
df['password'] = df['password'].astype(str)
df['length'] = df['password'].apply(len)
df.head(10)

File exists: True size MB: 26.396300315856934
Rows: 2503184


,password,length
0,Passwords,9
1,wed8800,7
2,wed8705j,8
3,wed869cut451,12
4,wed859dole143,13
5,wed8390,7
6,wed8262006,10
7,wed822jake377,13
8,wed821,6
9,wed81800,8


In [4]:
import math
import re
import numpy as np
import pandas as pd

def shannon_entropy(s: str) -> float:
    if not s:
        return 0.0
    freq = {}
    for ch in s:
        freq[ch] = freq.get(ch, 0) + 1
    entropy = 0.0
    L = len(s)
    for c in freq.values():
        p = c / L
        entropy -= p * math.log2(p)
    return entropy

def max_consecutive_repeat(s: str) -> int:
    if not s:
        return 0
    max_run = 1
    curr = 1
    for i in range(1, len(s)):
        if s[i] == s[i-1]:
            curr += 1
            if curr > max_run:
                max_run = curr
        else:
            curr = 1
    return max_run

year_regex = re.compile(r'(19|20)\d{2}')

def extract_features(passwords):
    # Accepts list/Series of passwords or single string. Returns DataFrame of features.
    if isinstance(passwords, str):
        passwords = [passwords]
    pwd_series = pd.Series(passwords).astype(str)
    lengths = pwd_series.str.len()
    n_lower = pwd_series.apply(lambda s: sum(1 for c in s if c.islower()))
    n_upper = pwd_series.apply(lambda s: sum(1 for c in s if c.isupper()))
    n_digits = pwd_series.apply(lambda s: sum(1 for c in s if c.isdigit()))
    n_symbols = lengths - n_lower - n_upper - n_digits
    unique_chars = pwd_series.apply(lambda s: len(set(s)))
    entropy = pwd_series.apply(shannon_entropy)
    max_rep = pwd_series.apply(max_consecutive_repeat)
    has_year = pwd_series.apply(lambda s: bool(year_regex.search(s)))
    # Character-class fractions (avoid division by zero)
    frac_digits = (n_digits / lengths).fillna(0)
    frac_upper = (n_upper / lengths).fillna(0)
    frac_lower = (n_lower / lengths).fillna(0)
    frac_symbol = (n_symbols / lengths).fillna(0)
    # How many character classes used
    classes_used = pwd_series.apply(lambda s: sum([
        any(c.islower() for c in s),
        any(c.isupper() for c in s),
        any(c.isdigit() for c in s),
        any((not c.isalnum()) for c in s)
    ]))
    df_feat = pd.DataFrame({
        "length": lengths,
        "n_lower": n_lower,
        "n_upper": n_upper,
        "n_digits": n_digits,
        "n_symbols": n_symbols,
        "unique_chars": unique_chars,
        "entropy": entropy,
        "max_consec_repeat": max_rep,
        "has_year": has_year.astype(int),
        "frac_digits": frac_digits,
        "frac_upper": frac_upper,
        "frac_lower": frac_lower,
        "frac_symbol": frac_symbol,
        "classes_used": classes_used
    })
    return df_feat


In [5]:
# build a set of "common passwords" from the most frequent ones in the dataset (to penalize them)
top_common = set(df['password'].value_counts().head(20000).index)

def rule_score_and_label(pwd):
    # compute features for single password and produce score 0-100 and label
    feats = extract_features(pwd).iloc[0]
    length = feats['length']
    classes = feats['classes_used']
    entropy = feats['entropy']
    is_common = pwd in top_common

    # Length points
    if length < 6:
        len_pts = 0
    elif length < 9:
        len_pts = 10
    elif length < 13:
        len_pts = 20
    else:
        len_pts = 30

    variety_pts = min(30, int(classes) * 10)   # up to 30
    entropy_pts = min(40, int(entropy * 4))    # entropy roughly scaled to 0-40

    score = len_pts + variety_pts + entropy_pts

    # big penalty if exactly a common password
    if is_common:
        score -= 60

    # clamp
    score = max(0, min(100, score))

    # label thresholds (adjust as you like)
    if score < 30:
        label = "very weak"
    elif score < 50:
        label = "weak"
    elif score < 70:
        label = "medium"
    elif score < 85:
        label = "strong"
    else:
        label = "very strong"
    return score, label

# Apply to a small subset (to save time). If you want to label everything, remove head().
sample_for_label = df['password'].head(20000).copy()  # change size to label more data
scores_labels = sample_for_label.apply(lambda p: rule_score_and_label(p))
scores = scores_labels.apply(lambda x: x[0])
labels = scores_labels.apply(lambda x: x[1])

# Attach to the df subset (for training)
train_df = pd.DataFrame({'password': sample_for_label, 'score': scores, 'label': labels})
train_df['label'].value_counts()


,count
label,
very weak,19984
medium,8
weak,8


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

# Prepare X, y
X = extract_features(train_df['password'].values)
y = train_df['label'].values

# Train/test split (stratify to keep label proportions)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Simple baseline model (RandomForest)
clf = RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=42)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

# Save model and feature column names
joblib.dump({"model": clf, "feature_cols": X.columns.tolist()}, "pwd_strength_model.joblib")
print("Saved model to pwd_strength_model.joblib")


Accuracy: 0.96825
              precision    recall  f1-score   support

      medium       0.00      0.00      0.00         2
   very weak       1.00      0.97      0.98      3997
        weak       0.00      0.00      0.00         1

    accuracy                           0.97      4000
   macro avg       0.33      0.32      0.33      4000
weighted avg       1.00      0.97      0.98      4000

Confusion matrix:
[[   0    2    0]
 [  18 3873  106]
 [   0    1    0]]
Saved model to pwd_strength_model.joblib


In [7]:
import joblib
import numpy as np

obj = joblib.load("pwd_strength_model.joblib")
model = obj["model"]
feature_cols = obj["feature_cols"]

def predict_password_strength(password: str):
    feat = extract_features(password)
    # ensure column order
    feat = feat[feature_cols]
    probs = model.predict_proba(feat)[0]          # shape (n_classes,)
    classes = model.classes_
    best_idx = np.argmax(probs)
    best_label = classes[best_idx]
    best_prob = probs[best_idx]
    # return full dictionary of class->prob too
    probs_dict = {cls: float(p) for cls, p in zip(classes, probs)}
    return {"label": best_label, "confidence": float(best_prob), "probs": probs_dict}

# quick test
print(predict_password_strength("password123"))
print(predict_password_strength("Y7u$k9#2!AbcX"))


{'label': 'very weak', 'confidence': 1.0, 'probs': {'medium': 0.0, 'very weak': 1.0, 'weak': 0.0}}
{'label': 'very weak', 'confidence': 1.0, 'probs': {'medium': 0.0, 'very weak': 1.0, 'weak': 0.0}}


In [8]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(clf, X, y, cv=5, scoring='f1_macro', n_jobs=-1)
print("5-fold F1_macro:", scores, "mean:", scores.mean())


5-fold F1_macro: [0.32787024 0.36509237 0.34277827 0.33224647 0.3598858 ] mean: 0.3455746305176901


In [9]:
# create a fast lookup set for leaks
# Use the most frequent N passwords or the entire dataset (be careful with memory)
common_set = set(df['password'].value_counts().head(200000).index)  # choose size as needed

# Or if you want entire dataset exact matches (memory permitting):
# all_set = set(df['password'].unique())


In [10]:
def predict_password_strength_with_leak(password: str):
    feat = extract_features(password)
    feat = feat[feature_cols]
    probs = model.predict_proba(feat)[0]
    classes = model.classes_
    best_idx = probs.argmax()
    best_label = classes[best_idx]
    best_prob = float(probs[best_idx])
    probs_dict = {cls: float(p) for cls, p in zip(classes, probs)}
    is_in_common = password in common_set
    # If you have an all_set: is_in_dataset = password in all_set
    return {
        "label": best_label,
        "confidence": best_prob,
        "probs": probs_dict,
        "leaked": bool(is_in_common),
        "leak_reason": "common_password" if is_in_common else "not_found_local"
    }


checking Data breach

In [15]:
# Ask user to enter a password
user_password = input("Enter your password: ")

# Check strength
print("Strength:", predict_password_strength_with_leak(user_password))

# Check if leaked
if user_password in common_set: # Use the pre-computed common_set
    print("⚠️ This password was found in leaks!")
else:
    print("✅ This password was not found in leaks.")

Enter your password: Tough$3421912./.'sfddlasd
Strength: {'label': 'very weak', 'confidence': 1.0, 'probs': {'medium': 0.0, 'very weak': 1.0, 'weak': 0.0}, 'leaked': False, 'leak_reason': 'not_found_local'}
✅ This password was not found in leaks.
